# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides users in loading, exploring, and analyzing the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/) library. The dataset contains mass spectrometry data for proteins isolated from stimulated human mast cells.

### Dataset Source
The dataset is defined via a Croissant schema accessible at this URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)

# The metadata is an object; print out dataset-level information
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(dataset.metadata, 'version', 'N/A')}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', [])}")

## 2. Data Overview
Explore available record sets and their structure, referencing everything by its `@id`.

Below, we enumerate all record sets, and for each, their fields and field IDs.

In [ ]:
print("Available record sets (by @id):\n")
record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    print("No record sets found in metadata. Attempting to infer record sets from the actual data files...")
    # Some datasets may not explicitly declare record sets. Let's check from dataset.records()
    print("Querying all available logical record sets via dataset.records(record_set=...)\n")
    # Try to get a list of available record sets from the public interface
    _ = dataset.records  # Exposes .record_sets
    try:
        rs_ids = dataset.record_sets()
    except Exception:
        rs_ids = []
    for rsid in rs_ids:
        print(f"- {rsid}")
    # Use as fallback below
else:
    for record_set in record_sets:
        print(f"- {record_set['@id']}: {record_set.get('name', '')}")
    rs_ids = [rs['@id'] for rs in record_sets]

# For demonstration, let's attempt loading at least one record set
# If the record sets remain empty, try with None or default
if not rs_ids:
    print("\nNo explicit record sets found (Croissant @id). Suggest inspecting available DataFrames after extraction.")
else:
    print("\nFields and @id for each record set:\n")
    for rs in record_sets:
        print(f"Record set '@id': {rs['@id']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"  - Field: {field_id}")

## 3. Data Extraction
Load data from the main record set(s) into `pandas` DataFrames by referencing record set and field `@id` values.

If the record set IDs are not available in the metadata, attempt with defaults (first, main, or auto-discovered record set).

In [ ]:
# Attempt to obtain all record set ids
if not rs_ids:
    # Try to probe via the dataset public interface
    try:
        rs_ids = dataset.record_sets()
    except Exception:
        rs_ids = [None] # Try to fallback to None for "default"

dataframes = {}

for record_set_id in rs_ids:
    try:
        print(f"\nLoading data from record set: {record_set_id if record_set_id else '[default]'} ...")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id if record_set_id else 'default'] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Number of records: {len(df)}")
        display(df.head(2))
    except Exception as e:
        print(f"  Failed to load records from record set '{record_set_id}'. Error: {e}")

# Pick the main record set for subsequent EDA. Fallback to the first loaded.
main_record_set_id = rs_ids[0] if rs_ids else list(dataframes.keys())[0]

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group the data based on numeric and categorical fields. All columns will be referenced by their `@id`.

We begin by selecting a numeric field. This field's `@id` can be found in the DataFrame's columns (which correspond to field `@id`s).

In [ ]:
# List available columns in the main record set:
df = dataframes[main_record_set_id]
print(f"Available columns (@id):\n{df.columns.tolist()}")

# Let's select a numeric column likely present in protein datasets, e.g. 'cr:mw' or 'cr:molecular_weight'
possible_numeric_fields = [col for col in df.columns if any(x in col.lower() for x in ['mw', 'weight', 'count', 'abundance', 'coverage', 'peptide'])]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    numeric_field_id = df.columns[0]
    print(f"No matching numeric field found. Fallback to: {numeric_field_id}")

# Filter: Only proteins (rows) with the numeric field > threshold (e.g. mean)
mean_value = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
threshold = mean_value
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean):")
display(filtered_df.head(3))

# Normalize the numeric column
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} (mean=0, std=1):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

# Try grouping by a probable categorical field, e.g., 'cr:sample_id', 'cr:description', etc.
possible_group_fields = [col for col in df.columns if any(x in col.lower() for x in ['cell', 'sample', 'group', 'type', 'desc'])]
if possible_group_fields:
    group_field_id = possible_group_fields[0]
    print(f"Grouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nMean {numeric_field_id} by {group_field_id} (top 5 groups):")
    display(grouped_df.head())
else:
    print("No clear group field detected. Skipping grouping.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and its normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[numeric_field_id], kde=True, ax=axes[0])
axes[0].set_title(f'Distribution of {numeric_field_id}')
axes[0].set_xlabel(numeric_field_id)
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, color='orange', ax=axes[1])
axes[1].set_title(f'Normalized {numeric_field_id} (filtered)')
axes[1].set_xlabel(numeric_field_id + '_normalized')
plt.tight_layout()
plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR<sup>2</sup> dataset describing mass spectrometry analysis of extracellular vesicle proteins. We listed record sets and field IDs, loaded the records, and performed filtering, normalization, and grouping using columns referenced by their unique `@id`. Visualization showed the distribution of a key numeric field before and after normalization. This workflow can be extended for deeper proteomic data analysis.